<a href="https://colab.research.google.com/github/Mxnaal25/Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mxnaal25/Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking (specifically, Learning to Rank).

Why: We are not predicting an exact number of clicks (regression), nor are we just putting pages into simple "good/bad" buckets (classification). We need to sort 30,000 pages from highest potential upside to lowest, so the content team knows exactly which 20 pages to fix first. That is a ranking problem.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/Proxy: The ideal target is the actual future CTR after a rewrite, but we don't have that data yet.

The Proxy: Our proxy target is a binary label (1 or 0) representing "High Opportunity." We will label a page as 1 if it has high impressions (e.g., top 20%) AND a CTR significantly below the median for its ranking position. A label of 0 means it's either low traffic or already performing fine. This label comes from a defined rule.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric: Precision@K (e.g., Precision@20).

Since the content team can only fix a limited number of pages per week, we care about the top of the list. If we hand them a ranked list of 20 pages, Precision@20 measures how many of those 20 pages were actually high-opportunity targets (true positives). We want to minimize the false positives at the very top of the queue.



In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the real dataset from GitHub
url = "https://raw.githubusercontent.com/Mxnaal25/Starter-Notebooks/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# --- UPDATED COLUMN NAMES ---
imp_col = 'impressions_90d'
ctr_col = 'ctr'

# Calculate the proxy target:
# High Opportunity = Top 20% in 90-day impressions AND CTR < 0.5% (which is 0.5 in this dataset)
imp_threshold = df[imp_col].quantile(0.80)
df['high_opportunity'] = ((df[imp_col] >= imp_threshold) & (df[ctr_col] < 0.5)).astype(int)

print("Label distribution (0 = No, 1 = Yes for High Opportunity):")
print(df['high_opportunity'].value_counts())


Label distribution (0 = No, 1 = Yes for High Opportunity):
high_opportunity
0    25166
1     4834
Name: count, dtype: int64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique web page (over a specific time window).

Below, I display the dataframe to show the unit of analysis and the newly created target column.



In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis with the target column
# We select a few relevant columns to make it easy to read
# Show the unit of analysis with the target column
# content_id is the unique page identifier in this dataset
cols_to_show = ['content_id', 'impressions_90d', 'ctr', 'high_opportunity']
if 'page_url' in df.columns: # Just in case they add URLs later
    cols_to_show = ['page_url'] + cols_to_show

print("Here is a sample of the dataframe (one row = one page):")
display(df[cols_to_show].head(5))

print(f"\nTotal rows (pages) in the dataset: {len(df)}")


Here is a sample of the dataframe (one row = one page):


,content_id,impressions_90d,ctr,high_opportunity
0,content_304f48230142,3803,0.76,0
1,content_a1fb4e703a9e,15320,0.05,1
2,content_9aa793d4d895,12581,0.09,1
3,content_331d6c4de07b,11751,0.49,1
4,content_d99b7a2d90ca,19140,0.13,1



Total rows (pages) in the dataset: 30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Right now, a simple fixed rule works okay: "Rank by high impressions AND CTR < 2%."

But a fixed rule fails when we try to scale or get more accurate. What if a page has a high CTR but it's ranking in position #1, meaning it should have a massive CTR? A fixed rule can't calculate the complex, non-linear relationships between impressions, position, keyword type, and content length.

A Machine Learning model (like XGBoost or a Random Forest) can look at dozens of features at once and learn the subtle patterns of what makes a page overperform or underperform relative to its specific situation. It moves us from a blunt instrument to a precision scalpel.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# To prove why a fixed rule is messy, let's look at pages that break a simple rule:
# High CTR but VERY low impressions (These look "good" by CTR, but nobody sees them)
# To prove why a fixed rule is messy, let's look at pages that break a simple rule:
# High CTR but VERY low impressions (These look "good" by CTR, but nobody sees them)
misleading_pages = df[(df['ctr'] > 0.05) & (df['impressions_90d'] < 100)]

print(f"Number of pages with 'great' CTR (>5%) but almost no traffic (<100 views over 90 days): {len(misleading_pages)}")
print("This is why ML is needed: a fixed rule might flag these as good, but they generate zero actual business value.")

Number of pages with 'great' CTR (>5%) but almost no traffic (<100 views over 90 days): 858
This is why ML is needed: a fixed rule might flag these as good, but they generate zero actual business value.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.